<a href="https://colab.research.google.com/github/raki-rankawat/thesis-v1/blob/master/Model_KD_Combined.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Model_KD_Combined
### Knowledge Distillation sweep — 2 Teachers × 2 Students × 2 Init Strategies = 8 Runs

| Role | Model |
|------|-------|
| Teacher A | VGG_Pretrained |
| Teacher B | ResNet_Pretrained |
| Student A | MobileNetV2 |
| Student B | MobileNetV3 |

**Per-run pipeline:**
1. Train student under KD (fine-tune from seed checkpoint OR from scratch)
2. Export FP32 ONNX → INT8 QDQ → compute NSE → flag quant-compatibility
3. Save record to `kd_combined_records.json` on Drive after every run

**Resume support:** re-running this notebook skips already-completed experiments automatically.  
Checkpoints that fail the NSE gate (`< 0.95`) are flagged and must not be passed to `Pipeline_Quantz`.

In [ ]:
# ── Mount Drive & load utils ────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import sys, shutil, os
UTILS_SRC = "/content/drive/My Drive/stm32-thesis/utils"
if os.path.exists(UTILS_SRC):
    shutil.copytree(UTILS_SRC, "/content/utils", dirs_exist_ok=True)
    sys.path.insert(0, "/content")
    print("✅ utils loaded from Drive")
else:
    sys.path.insert(0, "/content")
    print("⚠️  Place utils/ at: My Drive/stm32-thesis/utils/")

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────
import os, json, math, time
import numpy as np
import pandas as pd
import torch

from pathlib import Path

from utils.dataset import prepare_dataset, get_loaders, get_test_loader
from utils.models  import (
    VGG_Pretrained, ResNet_Pretrained,
    VWW_MobileNetV2, VWW_MobileNetV3,
)
from utils.train import setup_device, set_seed, evaluate, train_kd

device = setup_device(seed=41)

In [ ]:
# ── Dataset ──────────────────────────────────────────────────────────
prepare_dataset()
train_loader, val_loader = get_loaders(batch_size=64, augmentation="standard")
test_loader              = get_test_loader(batch_size=1)
print("✅ Dataset ready")

In [ ]:
# ── Config — edit here ───────────────────────────────────────────────
SAVE_DIR   = "/content/drive/My Drive/stm32-thesis/checkpoints"
EXPORT_DIR = Path("/content/drive/My Drive/stm32-thesis/exports")
SHARED_DIR = EXPORT_DIR / "shared"

# ── Teacher checkpoints ───────────────────────────────────────────────
TEACHER_VGG_CKPT    = f"{SAVE_DIR}/vgg_pretrained_seed_85.pth"
TEACHER_RESNET_CKPT = f"{SAVE_DIR}/resnet_pretrained_seed_85.pth"

# ── Student seed checkpoints (warm-start init) ────────────────────────
MV2_CKPT = f"{SAVE_DIR}/mobilenetv2_seed_74.pth"
MV3_CKPT = f"{SAVE_DIR}/mobilenetv3_seed_74.pth"

# ── Active teachers ───────────────────────────────────────────────────
ACTIVE_TEACHERS = {
    "vgg",        # VGG_Pretrained  — primary teacher
    "resnet",     # ResNet_Pretrained
}

# ── KD hyperparameters ────────────────────────────────────────────────
KD_TEMPERATURE = 4.0
KD_ALPHA       = 0.7
KD_EPOCHS      = 80
KD_WD          = 1e-4
KD_PATIENCE    = 20

# Fine-tune LR is per-student — do not unify these
# MobileNetV2: 3e-4 worked (gave +1.60%). 1e-3 destroyed the checkpoint.
# MobileNetV3: 3e-4 flatlined at +0.00%. Needs 1e-3 to escape local minimum.
KD_LR_FT_MV2 = 3e-4
KD_LR_FT_MV3 = 1e-3
KD_LR_SCR    = 1e-3   # scratch always 1e-3 regardless of student

# ── Quant gate ────────────────────────────────────────────────────────
NSE_THRESHOLD = 0.95

# ── Records JSON — resume support ─────────────────────────────────────
RECORDS_JSON = f"{SAVE_DIR}/kd_combined_records.json"

print("✅ Config loaded")
print(f"Active teachers : {ACTIVE_TEACHERS}")
print(f"NSE threshold   : {NSE_THRESHOLD}")

In [ ]:
# ── Quantisation helpers ─────────────────────────────────────────────
!pip -q install onnx onnxruntime onnxruntime-tools

import onnx
import onnxruntime as ort
from onnxruntime.quantization import quantize_static, QuantType, QuantFormat, CalibrationDataReader


def export_onnx(model, path):
    if Path(path).exists():
        print(f"    ⏭️  FP32 ONNX exists"); return
    model.eval()
    dummy = torch.randn(1, 3, 96, 96, device=device)
    torch.onnx.export(
        model, dummy, str(path),
        input_names=["input"], output_names=["logits"],
        export_params=True, opset_version=18,
        do_constant_folding=True,
        dynamic_axes={"input": {0: "batch_size"}, "logits": {0: "batch_size"}},
        dynamo=False,
    )
    onnx.checker.check_model(str(path), full_check=False)
    print(f"    ✅ FP32 ONNX saved")


def save_calib_npz(path, n=200):
    if Path(path).exists():
        print(f"    ⏭️  Calib data exists"); return
    xs = []
    with torch.no_grad():
        for i, (x, _) in enumerate(train_loader):
            if i >= n: break
            xs.append(x.numpy().astype("float32")[0])
    np.savez(path, input=np.stack(xs))
    print(f"    ✅ Calib data saved ({n} samples)")


def generate_shared_test_files(n=200):
    SHARED_DIR.mkdir(parents=True, exist_ok=True)
    inp_p = SHARED_DIR / "test_input.npz"
    lbl_p = SHARED_DIR / "test_labels.npz"
    if inp_p.exists() and lbl_p.exists():
        print(f"    ⏭️  Shared test files exist")
        return inp_p, lbl_p
    inputs, labels = [], []
    for i, (x, y) in enumerate(test_loader):
        if i >= n: break
        inputs.append(x.numpy().astype("float32")[0])
        labels.append(int(y.item()))
    np.savez(inp_p, input=np.stack(inputs))
    np.savez(lbl_p, label=np.array(labels, dtype="int32"))
    print(f"    ✅ Shared test files saved ({n} samples)")
    return inp_p, lbl_p


class CalibReader(CalibrationDataReader):
    def __init__(self, npz_path):
        self.data = np.load(npz_path)["input"].astype("float32")
        self.i = 0
    def get_next(self):
        if self.i >= len(self.data): return None
        out = {"input": self.data[self.i:self.i+1]}; self.i += 1; return out
    def rewind(self): self.i = 0


def quantize_int8(fp32_path, calib_path, int8_path):
    if Path(int8_path).exists():
        print(f"    ⏭️  INT8 ONNX exists"); return
    quantize_static(
        model_input=str(fp32_path),
        model_output=str(int8_path),
        calibration_data_reader=CalibReader(calib_path),
        quant_format=QuantFormat.QDQ,
        activation_type=QuantType.QInt8,
        weight_type=QuantType.QInt8,
        per_channel=True,
    )
    print(f"    ✅ INT8 QDQ ONNX saved")


def compute_nse(fp32_path, int8_path, input_npz):
    inputs = np.load(input_npz)["input"]
    s32 = ort.InferenceSession(str(fp32_path), providers=["CPUExecutionProvider"])
    s8  = ort.InferenceSession(str(int8_path), providers=["CPUExecutionProvider"])
    fp32_outs, int8_outs = [], []
    for i in range(len(inputs)):
        sample = inputs[i:i+1]
        fp32_outs.append(s32.run(["logits"], {"input": sample})[0][0])
        int8_outs.append(s8.run(["logits"],  {"input": sample})[0][0])
    fp32_outs = np.array(fp32_outs)
    int8_outs = np.array(int8_outs)
    num = np.sum((fp32_outs - int8_outs) ** 2)
    den = np.sum((fp32_outs - fp32_outs.mean()) ** 2)
    return float(1 - num / den)


print("✅ Quantisation helpers loaded")

In [ ]:
# ── Pre-flight ───────────────────────────────────────────────────────
print("Checking teacher checkpoints...")
for name, path in [("VGG_Pretrained", TEACHER_VGG_CKPT),
                   ("ResNet_Pretrained", TEACHER_RESNET_CKPT)]:
    ok = os.path.exists(path)
    print(f"  {'✅' if ok else '⚠️ (inactive ok)'}  {name}: {path}")

print("\nChecking student seed checkpoints...")
for name, path in [("MobileNetV2", MV2_CKPT), ("MobileNetV3", MV3_CKPT)]:
    ok = os.path.exists(path)
    print(f"  {'✅' if ok else '❌'}  {name}: {path}")
    if not ok:
        raise FileNotFoundError(f"Missing seed checkpoint: {path}")

EXPORT_DIR.mkdir(parents=True, exist_ok=True)
SHARED_INPUT_NPZ, _ = generate_shared_test_files()
print("\n✅ Pre-flight passed")

In [ ]:
# ── Load active teachers + student baselines ─────────────────────────
teacher_vgg    = None
teacher_resnet = None
vgg_acc        = None
resnet_acc     = None

if "vgg" in ACTIVE_TEACHERS:
    teacher_vgg = VGG_Pretrained().to(device)
    teacher_vgg.load_state_dict(torch.load(TEACHER_VGG_CKPT, map_location=device))
    teacher_vgg.eval()
    for p in teacher_vgg.parameters(): p.requires_grad = False
    vgg_acc = evaluate(teacher_vgg, val_loader, device)
    print(f"Teacher VGG_Pretrained    : {vgg_acc*100:.2f}%")
else:
    print("⏭️  VGG teacher inactive")

if "resnet" in ACTIVE_TEACHERS:
    teacher_resnet = ResNet_Pretrained().to(device)
    teacher_resnet.load_state_dict(torch.load(TEACHER_RESNET_CKPT, map_location=device))
    teacher_resnet.eval()
    for p in teacher_resnet.parameters(): p.requires_grad = False
    resnet_acc = evaluate(teacher_resnet, val_loader, device)
    print(f"Teacher ResNet_Pretrained : {resnet_acc*100:.2f}%")
else:
    print("⏭️  ResNet teacher inactive")

# Student baselines
_tmp = VWW_MobileNetV2().to(device)
_tmp.load_state_dict(torch.load(MV2_CKPT, map_location=device))
baseline_mv2 = evaluate(_tmp, val_loader, device); del _tmp

_tmp = VWW_MobileNetV3().to(device)
_tmp.load_state_dict(torch.load(MV3_CKPT, map_location=device))
baseline_mv3 = evaluate(_tmp, val_loader, device); del _tmp

BASELINES = {"MobileNetV2": baseline_mv2, "MobileNetV3": baseline_mv3}

print(f"\nStudent MobileNetV2 baseline : {baseline_mv2*100:.2f}%")
print(f"Student MobileNetV3 baseline : {baseline_mv3*100:.2f}%")

In [ ]:
# ── KD sweep — 2 teachers × 2 students × 2 init = 8 runs ────────────
# Resume: loads existing records from Drive, skips completed runs.

# ── Load existing progress ────────────────────────────────────────────
if os.path.exists(RECORDS_JSON):
    with open(RECORDS_JSON) as _f:
        records = json.load(_f)
    print(f"✅ Loaded {len(records)} existing records from Drive")
    for _r in records:
        print(f"   ↩️  {_r['run_key']:<25}  val_acc: {_r['val_acc_%']:.2f}%  "
              f"NSE: {_r['NSE']}  quant_ok: {_r['quant_ok']}")
else:
    records = []
    print("No existing records found — starting fresh")

print()

def _already_done(run_key):
    return any(r["run_key"] == run_key for r in records)

# ── Experiment definitions ────────────────────────────────────────────
# (run_key, teacher_key, teacher_model, student_name, student_cls,
#  init_strategy, student_seed_ckpt, lr)
EXPERIMENTS = [
    ("vgg_mv2_ft",         "vgg",    lambda: teacher_vgg,    "MobileNetV2", VWW_MobileNetV2, "ft",      MV2_CKPT, KD_LR_FT_MV2),
    ("vgg_mv2_scratch",    "vgg",    lambda: teacher_vgg,    "MobileNetV2", VWW_MobileNetV2, "scratch",  None,    KD_LR_SCR),
    ("vgg_mv3_ft",         "vgg",    lambda: teacher_vgg,    "MobileNetV3", VWW_MobileNetV3, "ft",      MV3_CKPT, KD_LR_FT_MV3),
    ("vgg_mv3_scratch",    "vgg",    lambda: teacher_vgg,    "MobileNetV3", VWW_MobileNetV3, "scratch",  None,    KD_LR_SCR),
    ("resnet_mv2_ft",      "resnet", lambda: teacher_resnet, "MobileNetV2", VWW_MobileNetV2, "ft",      MV2_CKPT, KD_LR_FT_MV2),
    ("resnet_mv2_scratch", "resnet", lambda: teacher_resnet, "MobileNetV2", VWW_MobileNetV2, "scratch",  None,    KD_LR_SCR),
    ("resnet_mv3_ft",      "resnet", lambda: teacher_resnet, "MobileNetV3", VWW_MobileNetV3, "ft",      MV3_CKPT, KD_LR_FT_MV3),
    ("resnet_mv3_scratch", "resnet", lambda: teacher_resnet, "MobileNetV3", VWW_MobileNetV3, "scratch",  None,    KD_LR_SCR),
]

for (run_key, teacher_key, teacher_fn, student_name,
     student_cls, init_strategy, seed_ckpt, lr) in EXPERIMENTS:

    # ── Skip inactive teacher ─────────────────────────────────────────
    if teacher_key not in ACTIVE_TEACHERS:
        print(f"⏭️  {run_key:<25} — teacher '{teacher_key}' inactive")
        continue

    # ── Skip if already done ──────────────────────────────────────────
    if _already_done(run_key):
        print(f"⏭️  {run_key:<25} — already done, skipping")
        continue

    teacher = teacher_fn()
    baseline = BASELINES[student_name]
    out_pth  = f"{SAVE_DIR}/{run_key}.pth"

    print(f"\n{'═'*60}")
    print(f"  {run_key}  |  {student_name}  |  init: {init_strategy}")
    print(f"  student baseline: {baseline*100:.2f}%  |  lr: {lr}")
    print(f"{'═'*60}")

    # ── Build student ─────────────────────────────────────────────────
    student = student_cls().to(device)
    if init_strategy == "ft":
        student.load_state_dict(torch.load(seed_ckpt, map_location=device))
        print(f"  Warm-start from seed checkpoint")
    else:
        print(f"  Random init (scratch)")

    # ── KD training ───────────────────────────────────────────────────
    t0 = time.time()
    val_acc, _ = train_kd(
        student      = student,
        teacher      = teacher,
        train_loader = train_loader,
        val_loader   = val_loader,
        device       = device,
        epochs       = KD_EPOCHS,
        lr           = lr,
        weight_decay = KD_WD,
        temperature  = KD_TEMPERATURE,
        alpha        = KD_ALPHA,
        patience     = KD_PATIENCE,
        save_path    = out_pth,
    )
    elapsed = (time.time() - t0) / 60
    delta   = (val_acc - baseline) * 100
    print(f"\n  After KD     : {val_acc*100:.2f}%  delta: {delta:+.2f}%  ({elapsed:.1f} min)")
    print(f"  Checkpoint   : {run_key}.pth")

    # ── Quantisation ──────────────────────────────────────────────────
    out_dir    = EXPORT_DIR / run_key
    out_dir.mkdir(parents=True, exist_ok=True)
    fp32_path  = out_dir / "model_fp32.onnx"
    calib_path = out_dir / "calib_train.npz"
    int8_path  = out_dir / "model_int8_qdq.onnx"

    # Reload best checkpoint saved by train_kd
    student_best = student_cls().to(device)
    student_best.load_state_dict(torch.load(out_pth, map_location=device))
    student_best.eval()

    export_onnx(student_best, fp32_path)
    save_calib_npz(calib_path)
    try:
        quantize_int8(fp32_path, calib_path, int8_path)
        nse    = compute_nse(fp32_path, int8_path, SHARED_INPUT_NPZ)
        nse_ok = nse >= NSE_THRESHOLD
        quant_error = None
    except Exception as e:
        nse    = float("nan")
        nse_ok = False
        quant_error = str(e)
        print(f"    ⚠️  Quantisation failed: {e}")

    status = "✅ deploy" if nse_ok else "⛔ skip Pipeline_Quantz"
    print(f"  NSE          : {nse:.4f}  {status}")

    records.append({
        "run_key"      : run_key,
        "teacher"      : teacher_key,
        "student"      : student_name,
        "init"         : init_strategy,
        "baseline_%"   : round(baseline * 100, 2),
        "val_acc_%"    : round(val_acc * 100, 2),
        "delta_%"      : round(delta, 2),
        "NSE"          : round(nse, 4) if not math.isnan(nse) else None,
        "quant_ok"     : nse_ok,
        "quant_error"  : quant_error,
        "ckpt"         : out_pth,
        "int8_path"    : str(int8_path) if nse_ok else None,
    })

    # ── Save progress immediately ──────────────────────────────────────
    with open(RECORDS_JSON, "w") as _f:
        json.dump(records, _f, indent=2)
    print(f"  💾 Progress saved ({len(records)} records total)")

    del student, student_best

print("\n\n✅ KD sweep complete (or all already done).")
print(f"   Records JSON: {RECORDS_JSON}")

In [ ]:
# ── Results table ────────────────────────────────────────────────────
# Reload from JSON so this cell works even across sessions
if os.path.exists(RECORDS_JSON):
    with open(RECORDS_JSON) as _f:
        records = json.load(_f)

df = pd.DataFrame(records)
df = df.sort_values(["student", "teacher", "init"]).reset_index(drop=True)

W = 95
print("=" * W)
print("KNOWLEDGE DISTILLATION RESULTS")
print("=" * W)

display_cols = ["run_key", "teacher", "student", "init",
                "baseline_%", "val_acc_%", "delta_%", "NSE", "quant_ok"]
print(df[display_cols].to_string(index=False))
print("=" * W)

# ── Per-student summary ───────────────────────────────────────────────
for student in ["MobileNetV2", "MobileNetV3"]:
    sub = df[df["student"] == student]
    if sub.empty: continue
    baseline = sub["baseline_%"].iloc[0]
    print(f"\n{student} (baseline: {baseline:.2f}%)")
    print(f"  {'Run':<25} {'Val Acc':>9}  {'Delta':>8}  {'NSE':>7}  Status")
    print(f"  {'-'*65}")
    for _, row in sub.iterrows():
        nse_str = f"{row['NSE']:.4f}" if row["NSE"] is not None else "  ERROR"
        gate    = "✅" if row["quant_ok"] else "⛔"
        print(f"  {row['run_key']:<25} {row['val_acc_%']:>8.2f}%  "
              f"{row['delta_%']:>+7.2f}%  {nse_str}  {gate}")

# ── NSE gate summary ──────────────────────────────────────────────────
print(f"\n{'─'*W}")
print("Quantisation gate (NSE ≥ 0.95):")
for _, row in df.iterrows():
    gate    = "✅ PASS — safe for Pipeline_Quantz" if row["quant_ok"] else "⛔ FAIL — exclude from Pipeline_Quantz"
    nse_str = f"{row['NSE']:.4f}" if row["NSE"] is not None else "ERROR"
    print(f"  {row['run_key']:<25}  NSE={nse_str}  {gate}")

# ── Best per student (quant-compatible only) ──────────────────────────
print(f"\n🏆 Best per student (quant-compatible only):")
deployable = df[df["quant_ok"] == True]
if len(deployable) == 0:
    print("  ⚠️  No quant-compatible checkpoints found.")
else:
    for student in ["MobileNetV2", "MobileNetV3"]:
        sub = deployable[deployable["student"] == student]
        if sub.empty:
            print(f"  {student}: no deployable checkpoint"); continue
        best = sub.loc[sub["val_acc_%"].idxmax()]
        print(f"  {student}: {best['run_key']:<25}  "
              f"{best['val_acc_%']:.2f}%  (delta {best['delta_%']:+.2f}%  NSE={best['NSE']:.4f})")

# ── Deployable list ───────────────────────────────────────────────────
print(f"\n📋 Deployable checkpoints (safe to pass to Pipeline_Quantz):")
dep_cols = ["run_key", "student", "init", "val_acc_%", "NSE", "int8_path", "ckpt"]
deployable_df = df[df["quant_ok"] == True][dep_cols]
if len(deployable_df) == 0:
    print("  None")
else:
    print(deployable_df.to_string(index=False))

In [ ]:
# ── Comparison chart — KD results vs baseline ────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

if os.path.exists(RECORDS_JSON):
    with open(RECORDS_JSON) as _f:
        records = json.load(_f)
df = pd.DataFrame(records)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle("Knowledge Distillation — Accuracy vs Baseline", fontsize=14, fontweight="bold")

for ax, student in zip(axes, ["MobileNetV2", "MobileNetV3"]):
    sub      = df[df["student"] == student].sort_values(["teacher", "init"])
    baseline = sub["baseline_%"].iloc[0] if len(sub) > 0 else 0

    run_labels = sub["run_key"].tolist()
    val_accs   = sub["val_acc_%"].tolist()
    quant_ok   = sub["quant_ok"].tolist()
    nses       = sub["NSE"].tolist()

    x      = np.arange(len(run_labels))
    colors = ["steelblue" if ok else "tomato" for ok in quant_ok]
    bars   = ax.bar(x, val_accs, color=colors, alpha=0.85, width=0.6)

    ax.axhline(baseline, color="gray", linestyle="--", linewidth=1.5,
               label=f"Baseline {baseline:.2f}%")

    # Annotate bars with NSE
    for i, (bar, nse, ok) in enumerate(zip(bars, nses, quant_ok)):
        nse_str = f"NSE={nse:.3f}" if nse is not None else "err"
        color   = "green" if ok else "red"
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                nse_str, ha="center", va="bottom", fontsize=7.5, color=color)
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()/2,
                f"{val_accs[i]:.2f}%", ha="center", va="center",
                fontsize=8, color="white", fontweight="bold")

    pass_patch = mpatches.Patch(color="steelblue", alpha=0.85, label="✅ NSE pass")
    fail_patch = mpatches.Patch(color="tomato",    alpha=0.85, label="⛔ NSE fail")
    ax.legend(handles=[ax.get_legend_handles_labels()[0][0], pass_patch, fail_patch],
              fontsize=8)

    ax.set_title(f"{student}")
    ax.set_xticks(x)
    ax.set_xticklabels([l.replace("_", "\n") for l in run_labels], fontsize=8)
    ax.set_ylabel("Val accuracy (%)")
    ax.set_ylim(min(val_accs + [baseline]) - 2, max(val_accs + [baseline]) + 2)
    ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plot_path = "/content/drive/My Drive/stm32-thesis/kd_results_chart.png"
plt.savefig(plot_path, dpi=150)
plt.show()
print(f"✅ Chart saved → {plot_path}")

## Next steps

### ✅ Checkpoints safe for `Pipeline_Quantz`
Use the **deployable checkpoint list** printed above. Only pass `.pth` files where `quant_ok = True`.

### 🔁 Resume / re-run
- **Skip all done runs:** just re-run the sweep cell — completed runs are skipped automatically
- **Force one specific run:** delete its entry from `kd_combined_records.json` on Drive, then re-run
- **Force everything:** delete `kd_combined_records.json` entirely

### ⚙️ Enabling ResNet teacher
Set `ACTIVE_TEACHERS = {"vgg", "resnet"}` in the config cell.  
ResNet teacher runs are included in the sweep but skipped when `"resnet"` is not active.